# 07 — MENA Geopolitical Regime Filter

Builds a two-layer signal framework:

| Layer | Frequency | Source | Role |
|-------|-----------|--------|------|
| **Fast** | Daily | yfinance / FRED | What markets price *now* |
| **Slow** | Annual → daily ffill | IMF PCPS + World Bank WDI | What fundamentals *are* |

When both layers agree → amplify conviction.  
When they diverge → regime-change warning, reduce conviction.

---
**No API key required.** IMF and World Bank are both open/public.

**Coverage:**
- IMF: Dubai crude, EU gas, fertilizer index, wheat, food index, copper (2000–present)
- World Bank: Military spend, food imports, fertilizer use, energy imports, CPI for SAU/IRN/IRQ/EGY/ARE/JOR

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'specvel'))
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from geopolitical import GeopoliticalRegimeFilter, build_mena_stress_index, get_current_regime, REGIME_MULTIPLIERS
from adapters.imf        import IMFAdapter
from adapters.world_bank import WorldBankAdapter

plt.rcParams.update({'figure.facecolor':'#040810','axes.facecolor':'#040810','axes.edgecolor':'#1a2a4a',
    'axes.labelcolor':'#8899aa','xtick.color':'#8899aa','ytick.color':'#8899aa',
    'text.color':'#ccd9e8','grid.color':'#0d1f35','grid.linewidth':0.5,'font.family':'monospace'})

CYAN='#00d4ff'; AMBER='#f5a623'; GREEN='#00e87a'; RED='#ff4444'
START='2000-01-01'; END='2026-03-10'
print('Setup complete.')

## 1. Build Composite Stress Index

In [ ]:
grf    = GeopoliticalRegimeFilter(verbose=True)
report = grf.run_report(start=START, end=END)
index_df = report['index_df']
print(f'Index shape: {index_df.shape}')
print(index_df.tail(10))

## 2. Current Regime Snapshot

In [ ]:
current_regime=report['current_regime']; current_z=report['current_zscore']
current_date=report['current_date'];  multipliers=report['multipliers']
print(f'Date:    {current_date}\nRegime:  {current_regime}\nZ-score: {current_z:.3f}\n')
print('Signal multipliers by asset class:')
for asset, mult in sorted(multipliers.items(), key=lambda x:-x[1]):
    bar = '\u25b2'*max(0,int((mult-1)*20)) if mult>1 else '\u25bc'*max(0,int((1-mult)*20))
    print(f'  {asset:<12} {mult:.2f}x  {bar:<5}  {"AMPLIFY" if mult>1 else "DAMPEN" if mult<1 else "NEUTRAL"}')

## 3. IMF Commodity Data

In [ ]:
imf = IMFAdapter()
commodity_series = {}
for code, label in [('POILDUB','Dubai Crude'),('PNGASEU','EU Natural Gas'),
                    ('PFERT','Fertilizer Index'),('PWHEAMT','Wheat'),('PFOOD','Food Index')]:
    try:
        s = imf.fetch(code, START, END)
        commodity_series[label] = s.resample('YE').last()
        print(f'  {code:10} {label:20} {len(s):5} daily obs  latest={s.iloc[-1]:.1f}')
    except Exception as e:
        print(f'  {code}: FAILED — {e}')
imf_panel = pd.DataFrame(commodity_series)
print(f'IMF panel: {imf_panel.shape}')
print(imf_panel.tail(6))

## 4. World Bank MENA Indicators

In [ ]:
wb = WorldBankAdapter()
for ind, label in [('AG.CON.FERT.ZS','Fertilizer Use'),('MS.MIL.XPND.GD.ZS','Military Spend'),
                   ('TM.VAL.FOOD.ZS.UN','Food Imports'),('FP.CPI.TOTL.ZG','Inflation CPI')]:
    print(f'\n── {label} ──')
    try:
        df = wb.fetch_panel(ind, START, END)
        print(df.resample('YE').last().dropna(how='all').tail(6))
    except Exception as e:
        print(f'  FAILED: {e}')

## 5. Country Stress Heatmap

In [ ]:
breakdown = report.get('country_breakdown', None)
if breakdown is not None and not breakdown.empty:
    numeric = breakdown.drop(columns=['iso3'],errors='ignore').select_dtypes(include='number')
    fig, ax = plt.subplots(figsize=(10,4),facecolor='#040810'); ax.set_facecolor('#040810')
    cmap = LinearSegmentedColormap.from_list('stress',['#0a4a2a','#0d2b0d','#040810','#2a1a0a','#6b1a0a'])
    im = ax.imshow(numeric.values, cmap=cmap, aspect='auto', vmin=-2, vmax=2)
    ax.set_xticks(range(len(numeric.columns))); ax.set_xticklabels(numeric.columns, rotation=30, ha='right', fontsize=8)
    ax.set_yticks(range(len(numeric.index)));   ax.set_yticklabels(numeric.index, fontsize=9)
    for i in range(len(numeric.index)):
        for j in range(len(numeric.columns)):
            v = numeric.iloc[i,j]
            if not __import__('numpy').isnan(v):
                c = '#ff4444' if v>1 else '#44ff88' if v<-1 else '#8899aa'
                ax.text(j,i,f'{v:.1f}',ha='center',va='center',fontsize=7,color=c,fontweight='bold')
    plt.colorbar(im, ax=ax, label='Velocity Z-Score')
    ax.set_title('MENA Country Stress Breakdown', color=CYAN, fontsize=11, pad=12)
    plt.tight_layout(); plt.show()
else:
    print('Country breakdown not available.')

## 6. Full History Chart

In [ ]:
annual_idx = index_df.resample('YE').last().dropna(subset=['composite_zscore'])
fig, axes = plt.subplots(3,1,figsize=(14,10),facecolor='#040810',gridspec_kw={'height_ratios':[2,2,1]})
regime_col = {'HIGH_STRESS':RED,'ELEVATED':AMBER,'NEUTRAL':'#8899aa','SUPPRESSED':'#44aa66','LOW_STRESS':GREEN}

ax=axes[0]
ax.plot(annual_idx.index, annual_idx['commodity_stress'], color=AMBER, lw=2, label='Commodity (IMF)')
ax.plot(annual_idx.index, annual_idx['country_stress'],   color=CYAN,  lw=2, label='Country (WB)')
ax.axhline(0,color='#1a2a4a',lw=0.8,linestyle='--'); ax.legend(fontsize=8)
ax.set_title('Stress Components',color=CYAN,fontsize=10); ax.grid(True,alpha=0.3)

ax=axes[1]
z=annual_idx['composite_zscore']
ax.axhspan(1.0,3.0,alpha=0.08,color=RED); ax.axhspan(0.3,1.0,alpha=0.06,color=AMBER)
ax.axhspan(-0.3,0.3,alpha=0.03,color='grey'); ax.axhspan(-1.0,-0.3,alpha=0.06,color=GREEN)
ax.axhspan(-3.0,-1.0,alpha=0.08,color=GREEN)
for i in range(1,len(z)):
    zi=z.iloc[i]; col=RED if zi>1 else AMBER if zi>0.3 else GREEN if zi<-0.3 else '#8899aa'
    ax.plot(z.index[i-1:i+1],z.iloc[i-1:i+1],color=col,lw=2.5,alpha=0.9)
ax.axhline(0,color='#1a2a4a',lw=0.8,linestyle='--'); ax.set_ylim(-3,3)
ax.set_title('Composite Z-Score with Regime Bands',color=CYAN,fontsize=10); ax.grid(True,alpha=0.3)

ax=axes[2]
for i in range(len(annual_idx)):
    r=annual_idx['regime'].iloc[i]
    ax.bar(annual_idx.index[i],1,width=300,color=regime_col.get(r,'#8899aa'),alpha=0.7)
ax.set_yticks([0.5]); ax.set_yticklabels(['Regime'],fontsize=7)
ax.set_title('Regime Timeline',color=CYAN,fontsize=10); ax.grid(False)

for ax in axes:
    ax.set_facecolor('#040810'); ax.tick_params(colors='#8899aa',labelsize=8)
    [sp.set_edgecolor('#1a2a4a') for sp in ax.spines.values()]
fig.suptitle('MENA Geopolitical Stress Index (2000–2026)',color=CYAN,fontsize=12,y=0.98)
plt.tight_layout(); plt.show()

## 7. Two-Layer Signal — Wheat

In [ ]:
import yfinance as yf; from scipy.signal import savgol_filter

wheat_raw = yf.download('ZW=F',start=START,end=END,progress=False,auto_adjust=True)['Close'].squeeze().dropna()
def velocity(s,win=7):
    v=s.dropna().values.astype(float); w=max(win if win%2==1 else win+1,5)
    w=min(w,len(v)-1 if(len(v)-1)%2==1 else len(v)-2)
    if len(v)>=w: v=savgol_filter(v,window_length=w,polyorder=2)
    g=np.gradient(v); mx=np.abs(g).max()
    if mx>0: g/=(mx+1e-9)
    return pd.Series(g,index=s.dropna().index).reindex(s.index)

lw=np.log(wheat_raw); norm_w=(lw-lw.min())/(lw.max()-lw.min())
fin_vel=velocity(norm_w); roll=120
fin_z=(fin_vel-fin_vel.rolling(roll,min_periods=60).mean().shift(1))/fin_vel.rolling(roll,min_periods=60).std().shift(1).clip(lower=1e-9)

geo_z_daily=index_df['composite_zscore'].reindex(wheat_raw.index,method='ffill')
div_df=grf.compute_divergence(index_df,fin_z,'ZW=F')
n_div=div_df['divergence'].sum()
print(f'Divergence events: {n_div} days ({n_div/len(div_df):.1%})')
print(div_df['divergence_type'].value_counts())

In [ ]:
fig,axes=plt.subplots(3,1,figsize=(14,9),facecolor='#040810',gridspec_kw={'height_ratios':[2,1.5,1]})
ax=axes[0]; ax.plot(wheat_raw.index,wheat_raw.values,color=AMBER,lw=1.2,alpha=0.9)
ax.set_title('Wheat Futures (ZW=F)',color=CYAN,fontsize=10); ax.set_ylabel('$/bushel',color='#8899aa'); ax.grid(True,alpha=0.3)

ax=axes[1]
fz=fin_z.resample('ME').mean(); gz=geo_z_daily.resample('ME').mean()
ax.plot(fz.index,fz.values,color=AMBER,lw=1.5,label='Financial velocity z (fast)',alpha=0.8)
ax.plot(gz.index,gz.values,color=CYAN, lw=2.0,label='Geo stress z (slow)',alpha=0.9)
ax.axhline(0,color='#1a2a4a',lw=0.8,linestyle='--'); ax.set_ylim(-4,4)
ax.axhline(0.75,color=RED,lw=0.5,linestyle=':',alpha=0.5); ax.axhline(-0.75,color=RED,lw=0.5,linestyle=':',alpha=0.5)
ax.set_title('Two-Layer: Fast Financial vs Slow Fundamental',color=CYAN,fontsize=10)
ax.legend(fontsize=8); ax.grid(True,alpha=0.3)

ax=axes[2]
dm=div_df['divergence'].resample('ME').mean()
dt=div_df['divergence_type'].resample('ME').agg(lambda x:(x=='GEO_BULLISH_FIN_BEARISH').mean()-(x=='GEO_BEARISH_FIN_BULLISH').mean())
colors=[RED if v>0 else GREEN if v<0 else '#8899aa' for v in dt]
ax.bar(dt.index,dm.values,color=colors,width=25,alpha=0.7)
ax.set_title('Divergence Events',color=CYAN,fontsize=9); ax.grid(True,alpha=0.3)

for ax in axes:
    ax.set_facecolor('#040810'); ax.tick_params(colors='#8899aa',labelsize=8)
    [sp.set_edgecolor('#1a2a4a') for sp in ax.spines.values()]
fig.suptitle('Wheat — Two-Layer Signal Framework',color=CYAN,fontsize=12,y=0.99)
plt.tight_layout(); plt.show()

## 8. Apply Regime Multipliers to Signal DataFrame

In [ ]:
signal_df=pd.DataFrame({'zscore':fin_z,'ticker':'ZW=F',
    'signal':fin_z.apply(lambda z:'LONG' if z>=0.75 else('SHORT' if z<=-0.75 else 'NEUTRAL'))},index=fin_z.index)
adjusted_df=grf.apply_to_signals(signal_df,index_df,ticker_col='ticker',zscore_col='zscore')
adj=adjusted_df.dropna(subset=['geo_adjusted_zscore','zscore'])
adj=adj.copy(); adj['delta']=adj['geo_adjusted_zscore']-adj['zscore']
print('Regime distribution:'); print(adj['geo_regime'].value_counts())
print('\nMean z-score delta by regime:')
print(adj.groupby('geo_regime')['delta'].agg(['mean','std','count']).round(3))
changed=adj[adj['geo_multiplier']!=1.0]
print(f'\nConviction changed: {len(changed)}/{len(adj)} rows ({len(changed)/len(adj):.1%})')
print(changed[['signal','geo_regime','geo_multiplier','zscore','geo_adjusted_zscore']].tail(8))

## 9. Fertilizer Velocity → Wheat Forward Return (IC Test)

In [ ]:
from scipy.stats import spearmanr
from geopolitical import _velocity_annual, _zscore_rolling

fert_raw=imf.fetch('PFERT',START,END)
fert_annual=imf.normalize(fert_raw).resample('YE').last().dropna()
fert_vel=_velocity_annual(fert_annual); fert_z=_zscore_rolling(fert_vel)

wheat_annual=wheat_raw.resample('YE').last()
wheat_fwd=wheat_annual.pct_change(1).shift(-1)
combined=pd.DataFrame({'fert_z':fert_z,'wheat_fwd_1y':wheat_fwd.reindex(fert_z.index,method='nearest')}).dropna()

ic,p=spearmanr(combined['fert_z'],combined['wheat_fwd_1y'])
print(f'Fertilizer velocity z -> wheat 1Y forward return')
print(f'  Spearman IC: {ic:.3f}   p-value: {p:.4f}   n={len(combined)}')
sig='SIGNIFICANT' if abs(ic)>0.3 and p<0.05 else 'WEAK' if abs(ic)>0.15 else 'NOT SIGNIFICANT'
print(f'  -> {sig}')

In [ ]:
fig,ax=plt.subplots(figsize=(8,5),facecolor='#040810'); ax.set_facecolor('#040810')
sc=ax.scatter(combined['fert_z'],combined['wheat_fwd_1y']*100,c=combined.index.year,cmap='plasma',s=60,alpha=0.8,zorder=3)
zf=np.polyfit(combined['fert_z'],combined['wheat_fwd_1y']*100,1)
zl=np.linspace(combined['fert_z'].min(),combined['fert_z'].max(),100)
ax.plot(zl,np.polyval(zf,zl),color=CYAN,lw=1.5,linestyle='--',alpha=0.7)
ax.axhline(0,color='#1a2a4a',lw=0.8); ax.axvline(0,color='#1a2a4a',lw=0.8)
plt.colorbar(sc,ax=ax,label='Year')
ax.set_xlabel('IMF Fertilizer Index Velocity Z',color='#8899aa')
ax.set_ylabel('Wheat 1Y Forward Return (%)',color='#8899aa')
ax.set_title(f'Fertilizer Velocity -> Wheat Forward Return  (IC={ic:.3f}, p={p:.3f})',color=CYAN,fontsize=10)
[sp.set_edgecolor('#1a2a4a') for sp in ax.spines.values()]
ax.tick_params(colors='#8899aa',labelsize=8); ax.grid(True,alpha=0.3)
plt.tight_layout(); plt.show()

## 10. Run Full Backtest with Geo Module

In [ ]:
from backtest import run_all_tests
T1,T2,T3,T4,T5,T6,T7,T7_index = run_all_tests(include_geo=True, fast=True)
print('\nT7 — Geo Regime History:')
print(T7.to_string(index=False) if not T7.empty else 'Empty')

## 11. Summary Dashboard

In [ ]:
if not T7_index.empty:
    fig=plt.figure(figsize=(16,10),facecolor='#040810')
    gs=gridspec.GridSpec(2,2,figure=fig,hspace=0.4,wspace=0.3)
    ai=T7_index.resample('YE').last().dropna(subset=['composite_zscore'])

    ax1=fig.add_subplot(gs[0,0])
    ax1.plot(ai.index,ai['commodity_stress'],color=AMBER,lw=2,label='Commodity')
    ax1.plot(ai.index,ai['country_stress'],color=CYAN,lw=2,label='Country')
    ax1.axhline(0,color='#1a2a4a',lw=0.8,linestyle='--'); ax1.legend(fontsize=7)
    ax1.set_title('Stress Components',color=CYAN,fontsize=9); ax1.grid(True,alpha=0.3)

    ax2=fig.add_subplot(gs[0,1])
    z=ai['composite_zscore']
    bc=[RED if v>1 else AMBER if v>0.3 else GREEN if v<-0.3 else '#8899aa' for v in z]
    ax2.bar(z.index,z.values,color=bc,width=300,alpha=0.8)
    ax2.axhline(0,color='#1a2a4a',lw=0.8); ax2.set_title('Composite Z-Score',color=CYAN,fontsize=9); ax2.grid(True,alpha=0.3)

    ax3=fig.add_subplot(gs[1,0])
    wm=ai['regime'].apply(lambda r:grf.get_multiplier(r,'ZW=F'))
    cm=ai['regime'].apply(lambda r:grf.get_multiplier(r,'CL=F'))
    ax3.plot(ai.index,wm,color=AMBER,lw=2,label='Wheat ZW=F'); ax3.plot(ai.index,cm,color=CYAN,lw=2,label='Crude CL=F')
    ax3.axhline(1.0,color='#1a2a4a',lw=0.8,linestyle='--'); ax3.set_ylim(0.7,1.4)
    ax3.set_title('Signal Multiplier by Year',color=CYAN,fontsize=9); ax3.legend(fontsize=7); ax3.grid(True,alpha=0.3)

    ax4=fig.add_subplot(gs[1,1])
    rc={'HIGH_STRESS':RED,'ELEVATED':AMBER,'NEUTRAL':'#8899aa','SUPPRESSED':'#44aa66','LOW_STRESS':GREEN}
    rcounts=T7['regime'].value_counts()
    ax4.pie(rcounts.values,labels=rcounts.index,colors=[rc.get(r,'#8899aa') for r in rcounts.index],
            autopct='%1.0f%%',textprops={'color':'#ccd9e8','fontsize':7},pctdistance=0.75,startangle=90)
    ax4.set_title('Regime Distribution',color=CYAN,fontsize=9)

    for ax in [ax1,ax2,ax3]:
        ax.set_facecolor('#040810'); ax.tick_params(colors='#8899aa',labelsize=7)
        [sp.set_edgecolor('#1a2a4a') for sp in ax.spines.values()]
    ax4.set_facecolor('#040810')
    fig.suptitle('MENA Geopolitical Regime Filter — Dashboard',color=CYAN,fontsize=13,y=0.99)
    plt.show()
else: print('T7_index empty.')

---
## Summary

| Module | Source | Role |
|--------|--------|------|
| `IMFAdapter` | IMF PCPS | Commodity fundamental velocity |
| `WorldBankAdapter` | World Bank WDI | MENA country stress velocity |
| `GeopoliticalRegimeFilter` | Both | Composite index + regime label |

### Multiplier table

| Regime | Z | Wheat | Crude | Gas | Equities |
|--------|---|-------|-------|-----|----------|
| HIGH_STRESS | >+1.0 | 1.30× | 1.20× | 1.20× | 0.80× |
| ELEVATED | +0.3→+1.0 | 1.10× | 1.10× | 1.10× | 0.90× |
| NEUTRAL | ±0.3 | 1.00× | 1.00× | 1.00× | 1.00× |
| SUPPRESSED | -0.3→-1.0 | 0.90× | 0.90× | 0.90× | 1.10× |
| LOW_STRESS | <-1.0 | 0.80× | 0.80× | 0.80× | 1.20× |

```bash
# Standalone
python specvel/geopolitical.py --save results/geo_index.csv

# With backtest
python specvel/backtest.py --fast --include_geo
```